<a href="https://colab.research.google.com/github/simon-mellergaard/datavis/blob/main/Data/Indexing%20NLP%20algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Libraries needed**

In [2]:
import os
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
from huggingface_hub import notebook_login

notebook_login()

# **Load data**

In [5]:
import os
import sys
if 'google.colab' in sys.modules:
    %cd /content/
    # remove local directory if it already exists
    if os.path.isdir("datavis"):
        !rm -rf {"datavis"}
    !git clone https://github.com/simon-mellergaard/datavis.git
    %cd /content/datavis/Data

/content
Cloning into 'datavis'...
remote: Enumerating objects: 576, done.
remote: Counting objects: 100% (70/70), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 576 (delta 34), reused 42 (delta 16), pack-reused 506 (from 1)
Receiving objects: 100% (576/576), 40.41 MiB | 29.56 MiB/s, done.
Resolving deltas: 100% (324/324), done.
/content/datavis/Data


In [6]:
path = "DATA_UFM_combined.xlsx"
df = pd.read_excel(path)

# **TESTING**

In [ ]:
COL_TITLE = "titel"
OUT_DIR = "/content/drive/MyDrive/DATA VIS TEST/"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = f"{OUT_DIR}/education_cluster_mapping.csv"
OUT_XLSX = f"{OUT_DIR}/education_cluster_mapping.xlsx"

df = df[df[COL_TITLE].astype(str).str.len() > 0].copy()

# Clusters
clusters = [
    ("Sundhed & Omsorg",
     "Uddannelser inden for sundhed, pleje, patientbehandling, terapi og omsorg."),
    ("Business, Økonomi & Ledelse",
     "Uddannelser om forretning, finans, marketing, økonomi og ledelse."),
    ("Ingeniør, IT & Data",
     "Uddannelser om ingeniørfag, software, data, informationsteknologi og produktion."),
    ("Samfund, Jura & Forvaltning",
     "Uddannelser om jura, politik, samfund, offentlig forvaltning og socialt arbejde."),
    ("Natur, Miljø & Fødevarer",
     "Uddannelser om naturvidenskab, miljø, landbrug, naturressourcer og fødevarer."),
    ("Uddannelse, Pædagogik & Social",
     "Uddannelser om undervisning, pædagogik, didaktik og sociale indsatser."),
    ("Kunst, Design, Medier & Kommunikation",
     "Uddannelser om kunst, arkitektur, design, medier, kommunikation og journalistik."),
]
cluster_labels = [c[0] for c in clusters]
label_texts = [f"{name}. {desc}" for name, desc in clusters]

# Model
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(MODEL_NAME, device=device)

label_emb = model.encode(
    label_texts,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True
)
titles = df[COL_TITLE].astype(str).str.strip().add(" uddannelse").tolist()
text_emb = model.encode(
    titles,
    batch_size=128,
    convert_to_tensor=True,
    normalize_embeddings=True
)

sims = text_emb @ label_emb.T  # [N, 7]
best_scores, best_idx = torch.max(sims, dim=1)

df["cluster_label"] = [cluster_labels[i] for i in best_idx.tolist()]
df["cluster_score"] = best_scores.detach().cpu().numpy()

# low-confidence bucket
THRESHOLD = 0.40
df.loc[df["cluster_score"] < THRESHOLD, "cluster_label"] = "Øvrige/Ukendt"

# Save
out = df[[COL_TITLE, "cluster_label", "cluster_score"]].copy()
out.to_csv(OUT_CSV, index=False)
out.to_excel(OUT_XLSX, index=False)

print(f"Saved CSV:   {OUT_CSV}")
print(f"Saved Excel: {OUT_XLSX}")
print("\nCounts per cluster:")
print(out["cluster_label"].value_counts())


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Saved CSV:   /content/drive/MyDrive/DATA VIS TEST//education_cluster_mapping.csv
Saved Excel: /content/drive/MyDrive/DATA VIS TEST//education_cluster_mapping.xlsx

Counts per cluster:
cluster_label
Ingeniør, IT & Data                      549
Uddannelse, Pædagogik & Social           533
Business, Økonomi & Ledelse              280
Kunst, Design, Medier & Kommunikation    228
Natur, Miljø & Fødevarer                 215
Sundhed & Omsorg                         211
Øvrige/Ukendt                             92
Samfund, Jura & Forvaltning               41
Name: count, dtype: int64


In [7]:
COL_TITLE = "titel"
OUT_DIR = "/content/drive/MyDrive/Colab_Notebooks"
os.makedirs(OUT_DIR, exist_ok=True)
OUT_CSV = f"{OUT_DIR}/education_cluster_mapping.csv"
OUT_XLSX = f"{OUT_DIR}/education_cluster_mapping.xlsx"
df = df[df[COL_TITLE].astype(str).str.len() > 0].copy()

# Clusters
clusters = [
    ("Byggeri - transport",
     "Uddannelser om byggeri, anlæg, infrastruktur, transport og logistik."),
    ("Design - kunst",
     "Uddannelser om design, kunsthåndværk, visuel kunst og grafisk design."),
    ("Ernæring - sundhed - omsorg",
     "Uddannelser om ernæring, sundhed, pleje, behandling, terapi og omsorg."),
    ("Film - teater - musik",
     "Uddannelser om film, teater, scenekunst, musik og performancekunst."),
    ("Sikkerhed og forsvar",
     "Uddannelser om sikkerhed, forsvar, beredskab og militære forhold."),
    ("Handel - økonomi - markedsføring",
     "Uddannelser om handel, forretning, økonomi, finans og markedsføring."),
    ("IT - teknik - produktion",
     "Uddannelser om IT, teknik, ingeniørfag, produktion og teknologi."),
    ("Kultur - sprog - medier",
     "Uddannelser om kultur, sprog, kommunikation, medier og journalistik."),
    ("Natur - klima - miljø",
     "Uddannelser om naturvidenskab, klima, miljø, bæredygtighed og naturressourcer."),
    ("Pædagogik - psykologi - sociale forhold",
     "Uddannelser om pædagogik, psykologi, socialt arbejde og sociale indsatser."),
    ("Samfund - kontor - forvaltning",
     "Uddannelser om samfund, jura, politik, administration, kontor og forvaltning."),
    ("Undervisning",
     "Uddannelser om undervisning, læreruddannelser og didaktik."),
]

cluster_labels = [c[0] for c in clusters]
label_texts = [f"{name}. {desc}" for name, desc in clusters]

# Model
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)

label_emb = model.encode(
    label_texts,
    batch_size=32,
    convert_to_tensor=True,
    normalize_embeddings=True
)

titles = df[COL_TITLE].astype(str).str.strip().add(" uddannelse").tolist()
text_emb = model.encode(
    titles,
    batch_size=128,
    convert_to_tensor=True,
    normalize_embeddings=True
)

sims = text_emb @ label_emb.T  # [N, 12]
best_scores, best_idx = torch.max(sims, dim=1)

df["cluster_label"] = [cluster_labels[i] for i in best_idx.tolist()]
df["cluster_score"] = best_scores.detach().cpu().numpy()

# low-confidence bucket
THRESHOLD = 0.40
df.loc[df["cluster_score"] < THRESHOLD, "cluster_label"] = "Øvrige/Ukendt"

# Save
out = df[[COL_TITLE, "cluster_label", "cluster_score"]].copy()
out.to_csv(OUT_CSV, index=False)
out.to_excel(OUT_XLSX, index=False)

print(f"Saved CSV:   {OUT_CSV}")
print(f"Saved Excel: {OUT_XLSX}")
print("\nCounts per cluster:")
print(out["cluster_label"].value_counts())

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Saved CSV:   /content/drive/MyDrive/Colab_Notebooks/education_cluster_mapping.csv
Saved Excel: /content/drive/MyDrive/Colab_Notebooks/education_cluster_mapping.xlsx

Counts per cluster:
cluster_label
Undervisning                               480
IT - teknik - produktion                   447
Handel - økonomi - markedsføring           242
Natur - klima - miljø                      167
Ernæring - sundhed - omsorg                148
Design - kunst                             130
Byggeri - transport                        129
Kultur - sprog - medier                    113
Film - teater - musik                       86
Øvrige/Ukendt                               59
Pædagogik - psykologi - sociale forhold     53
Sikkerhed og forsvar                        53
Samfund - kontor - forvaltning              42
Name: count, dtype: int64
